#Hw2.1.ipynb

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests # Import requests library

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

url = "https://uk.wikipedia.org/wiki/Коефіцієнт_народжуваності_в_регіонах_України_(1950—2019)"

# Use requests to get the content, then pass it to pd.read_html
try:
    # Add User-Agent header to mimic a browser and avoid 403 Forbidden errors
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    tables = pd.read_html(response.text) # Pass the content of the response
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
    tables = [] # Assign an empty list or handle as appropriate

birth_df = None
for t in tables:
    # Assuming '2019' is a reliable indicator for the correct table
    if any('2019' in str(c) for c in t.columns):
        birth_df = t
        break # Stop after finding the first suitable table

# Fallback if no table with '2019' is found, use the last table
if birth_df is None and tables:
    birth_df = tables[-1]
elif birth_df is None:
    print("No tables found or suitable table could not be identified.")
    # Exit or raise an error if birth_df is essential for subsequent operations
    # For now, we'll let it proceed with an empty or problematic birth_df

# Proceed only if birth_df is not None
if birth_df is not None:
    if birth_df.columns[0] != 'region':
        birth_df = birth_df.rename(columns={birth_df.columns[0]: 'region'})

    display(birth_df.head())
    print("shape:", birth_df.shape)

    birth_df = birth_df.replace("—", np.nan)

    display(birth_df.dtypes)

    for col in birth_df.columns:
        if col != 'region':
            birth_df[col] = pd.to_numeric(birth_df[col], errors='coerce')

    display(birth_df.dtypes)

    print("missing share:")
    display(birth_df.isnull().mean())

    birth_df = birth_df.iloc[:-1, :]

    num_cols = birth_df.columns.drop('region', errors='ignore')
    birth_df[num_cols] = birth_df[num_cols].fillna(birth_df[num_cols].mean(numeric_only=True))

    print("missing share after fill:")
    display(birth_df.isnull().mean())

    # Ensure 'col_2019' is found before proceeding with calculations dependent on it
    col_2019_list = [c for c in birth_df.columns if '2019' in str(c)]
    if col_2019_list:
        col_2019 = col_2019_list[0]
        avg_2019 = birth_df[col_2019].mean()
        regions_above_avg_2019 = birth_df.loc[birth_df[col_2019] > avg_2019, 'region']
        print("Регіони > середнього у 2019:", regions_above_avg_2019.tolist())

        plt.figure(figsize=(12, 6))
        sns.barplot(x='region', y=col_2019, data=birth_df, palette='viridis')
        plt.xticks(rotation=90)
        plt.title('Народжуваність по регіонах України, 2019')
        plt.ylabel('Коефіцієнт народжуваності')
        plt.tight_layout()
        plt.show()

        top10_2019 = birth_df.sort_values(col_2019, ascending=False).head(10)
        plt.figure(figsize=(8, 5))
        sns.barplot(x='region', y=col_2019, data=top10_2019, palette='magma')
        plt.xticks(rotation=45, ha='right')
        plt.title('Топ-10 регіонів за народжуваністю у 2019')
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6, 4))
        sns.histplot(birth_df[col_2019], bins=10, kde=True, color='teal')
        plt.title('Розподіл коефіцієнта народжуваності у 2019')
        plt.xlabel('Коефіцієнт')
        plt.tight_layout()
        plt.show()
    else:
        print("Column for 2019 not found, skipping related visualizations.")

    col_2014_list = [c for c in birth_df.columns if '2014' in str(c)]
    if col_2014_list:
        col_2014 = col_2014_list[0]
        max_2014_region = birth_df.loc[birth_df[col_2014].idxmax(), 'region']
        print("Найвища народжуваність у 2014:", max_2014_region)
    else:
        print("Column for 2014 not found, skipping related calculations.")

    year_cols = [c for c in birth_df.columns if c != 'region']
    if year_cols:
        # Ensure all columns in year_cols are numeric before conversion
        numeric_year_cols = [col for col in year_cols if pd.api.types.is_numeric_dtype(birth_df[col])]
        if numeric_year_cols:
            year_nums = [int(str(c)[:4]) for c in numeric_year_cols]
            avg_by_year = birth_df[numeric_year_cols].mean(numeric_only=True)

            plt.figure(figsize=(10, 5))
            plt.plot(year_nums, avg_by_year.values, marker='o')
            plt.title('Середня народжуваність по Україні за роками')
            plt.xlabel('Рік')
            plt.ylabel('Середній коефіцієнт')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            sample_year_cols = numeric_year_cols[-10:] if len(numeric_year_cols) > 10 else numeric_year_cols
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=birth_df[sample_year_cols])
            plt.xticks(rotation=45)
            plt.title('Розподіл народжуваності по вибраних роках')
            plt.tight_layout()
            plt.show()
        else:
            print("No numeric year columns found for time series analysis.")
    else:
        print("No year columns found for time series analysis.")

Error fetching URL: 404 Client Error: Not Found for url: https://uk.wikipedia.org/wiki/%D0%9A%D0%BE%D0%B5%D1%84%D1%96%D1%86%D1%96%D1%94%D0%BD%D1%82_%D0%BD%D0%B0%D1%80%D0%BE%D0%B4%D0%B6%D1%83%D0%B2%D0%B0%D0%BD%D0%BE%D1%81%D1%82%D1%96_%D0%B2_%D1%80%D0%B5%D0%B3%D1%96%D0%BE%D0%BD%D0%B0%D1%85_%D0%A3%D0%BA%D1%80%D0%B0%D1%97%D0%BD%D0%B8_(1950%E2%80%942019)
No tables found or suitable table could not be identified.
